In [1]:
import nltk
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import string
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split
from sklearn  import svm



**Sources des données du fichier SMSSpamCollection.txt** 

[1] Almeida, T.A., Gómez Hidalgo, J.M., Yamakami, A. Contributions to the Study of SMS Spam Filtering: New Collection and Results. Proceedings of the 2011 ACM Symposium on Document Engineering (DOCENG'11), Mountain View, CA, USA, 2011.

[2] Gómez Hidalgo, J.M., Almeida, T.A., Yamakami, A. On the Validity of a New SMS Spam Collection. Proceedings of the 11th IEEE International Conference on Machine Learning and Applications (ICMLA'12), Boca Raton, FL, USA, 2012.

[3] Almeida, T.A., Gómez Hidalgo, J.M., Silva, T.P. Towards SMS Spam Filtering: Results under a New Dataset. International Journal of Information Security Science (IJISS), 2(1), 1-18, 2013.

http://dcomp.sor.ufscar.br/talmeida/smspamcollection/

In [12]:
data = pd.read_csv("../Data/SMSSpamCollection.txt", sep='\t', header=None)
data.columns = ['label', 'Content']
print(data)


     label                                            Content
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [13]:
data['Content_len'] = data['Content'].apply(lambda x: len(x) - x.count(" "))

print(data)

     label                                            Content  Content_len
0      ham  Go until jurong point, crazy.. Available only ...           92
1      ham                      Ok lar... Joking wif u oni...           24
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...          128
3      ham  U dun say so early hor... U c already then say...           39
4      ham  Nah I don't think he goes to usf, he lives aro...           49
...    ...                                                ...          ...
5567  spam  This is the 2nd time we have tried 2 contact u...          131
5568   ham               Will ü b going to esplanade fr home?           29
5569   ham  Pity, * was in mood for that. So...any other s...           48
5570   ham  The guy did some bitching but I acted like i'd...          100
5571   ham                         Rofl. Its true to its name           21

[5572 rows x 3 columns]


In [14]:
def count_punctuation(text):
    binary_array = [1 for ch in text if ch in string.punctuation] 
    nb_ponctuation = sum(binary_array)
    total = len(text) - text.count(" ")
    return round(nb_ponctuation/(total), 4)*100

 


In [15]:
data['punctuation_rate'] = data['Content'].apply(lambda x: count_punctuation(x))

print(data)


     label                                            Content  Content_len  \
0      ham  Go until jurong point, crazy.. Available only ...           92   
1      ham                      Ok lar... Joking wif u oni...           24   
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...          128   
3      ham  U dun say so early hor... U c already then say...           39   
4      ham  Nah I don't think he goes to usf, he lives aro...           49   
...    ...                                                ...          ...   
5567  spam  This is the 2nd time we have tried 2 contact u...          131   
5568   ham               Will ü b going to esplanade fr home?           29   
5569   ham  Pity, * was in mood for that. So...any other s...           48   
5570   ham  The guy did some bitching but I acted like i'd...          100   
5571   ham                         Rofl. Its true to its name           21   

      punctuation_rate  
0                 9.78  
1            

In [16]:
en_stopwords = nltk.corpus.stopwords.words('english')
ps = nltk.PorterStemmer()

def clean_email(email):
    result = "".join([word for word in email if word not in string.punctuation])
    tokens = re.split('\W+', result)
    text = [ps.stem(word) for word in tokens if word not in en_stopwords]
    return text



In [17]:

vectorisation_full = TfidfVectorizer(analyzer=clean_email)
vect_final = vectorisation_full.fit_transform(data['Content'])




In [18]:
all_data = pd.concat([pd.DataFrame(vect_final.toarray()), data['Content_len'], data['punctuation_rate']], axis=1)
print(all_data)

        0    1    2    3    4    5    6    7    8    9  ...  8185  8186  8187  \
0     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
1     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
2     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
3     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
4     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
...   ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   ...   ...   
5567  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
5568  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
5569  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
5570  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
5571  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   

      8188  8189      8190 

In [19]:
x_train, x_test, y_train, y_test = train_test_split(all_data, data['label'], test_size=0.2)

In [11]:
alg_svm= svm.SVC(kernel = 'linear')
alg_svm.fit(x_train, y_train)

SVC(kernel='linear')

In [22]:
 y_test

538      ham
1971     ham
973      ham
4029     ham
4480     ham
        ... 
5207     ham
205      ham
5057     ham
5228    spam
1269    spam
Name: label, Length: 1115, dtype: object

In [20]:
predictions = alg_svm.predict(x_test)

In [21]:
precision, recall, fscore, _ = score(y_test, predictions, pos_label='spam', average='binary')
print('Precision: {} / Recall: {} / Accuracy: {}'.format(round(precision, 3),
                                                        round(recall, 3),
                                                        round((predictions==y_test).sum() / len(predictions),3)))

Precision: 1.0 / Recall: 0.94 / Accuracy: 0.992


In [ ]:
print('Precision: {} / Recall: {} / Accuracy: {}'.format(round(precision, 3),
                                                        round(recall, 3),
                                                        round((predictions==y_test).sum() / len(predictions),3)))